# Enterprise Agent Governance Operating Model — Practical Lab

## Scenario
Build an enterprise onboarding workflow from **agent registration → risk assessment → testing → approval → deployment → continuous monitoring → recertification**.

The core lab is runnable without cloud credentials. The patterns are designed to map to enterprise registries, GRC workflows, policy engines and CI/CD systems.

In [ ]:
%pip install -q "pydantic>=2" pandas numpy pyyaml
print("Dependencies installed.")

In [ ]:
from pydantic import BaseModel, Field
from typing import Literal, Optional, Any
from datetime import datetime, timezone, timedelta
import pandas as pd, numpy as np, uuid, hashlib, json, yaml
def now(): return datetime.now(timezone.utc)
def sha(x): return hashlib.sha256(json.dumps(x,sort_keys=True,default=str).encode()).hexdigest()

## 1. Define a machine-readable Agent Card

In [ ]:
class Dependency(BaseModel):
    name:str
    kind:Literal["model","library","container","tool","mcp","data","api","agent"]
    version:str
    provider:str
    third_party:bool=False
    digest:Optional[str]=None
    trust:Literal["LOW","MEDIUM","HIGH"]="MEDIUM"

class AgentCard(BaseModel):
    agent_id:str
    name:str
    version:str
    business_owner:Optional[str]
    technical_owner:Optional[str]
    business_unit:str
    purpose:str
    autonomy:int = Field(ge=0,le=3)
    impact:int = Field(ge=1,le=4)
    access:int = Field(ge=1,le=4)
    irreversibility:int = Field(ge=1,le=4)
    data_sensitivity:int = Field(ge=1,le=4)
    external_actions:bool=False
    permissions:list[str]=[]
    dependencies:list[Dependency]=[]
    status:str="DRAFT"

procurement=AgentCard(
 agent_id="AG-002",name="Procurement Agent",version="3.2.0",
 business_owner="Procurement VP",technical_owner="AI Procurement Team",
 business_unit="Procurement",purpose="Bounded purchasing",
 autonomy=2,impact=3,access=3,irreversibility=2,data_sensitivity=2,
 external_actions=True,permissions=["vendor.read","po.prepare","po.create"],
 dependencies=[
   Dependency(name="foundation-model-x",kind="model",version="2026-07",provider="Vendor A",third_party=True,trust="HIGH"),
   Dependency(name="procurement-mcp",kind="mcp",version="1.4.2",provider="Internal Platform",digest="sha256:abc",trust="HIGH")
 ])
)
procurement

## 2. Enterprise registry

In [ ]:
registry={}
def register(card):
    key=f"{card.agent_id}:{card.version}"
    registry[key]=card.model_dump()
    return key
register(procurement)
pd.DataFrame(registry.values())[["agent_id","name","version","business_owner","status"]]

## 3. Ownership gate

In [ ]:
def ownership_gate(card):
    findings=[]
    if not card.business_owner: findings.append("MISSING_BUSINESS_OWNER")
    if not card.technical_owner: findings.append("MISSING_TECHNICAL_OWNER")
    return findings
ownership_gate(procurement)

## 4. Risk and autonomy classification

In [ ]:
def score(card):
    raw=card.autonomy*card.impact*card.access*card.irreversibility
    raw += card.data_sensitivity*2
    raw += 8 if card.external_actions else 0
    raw += sum(4 for d in card.dependencies if d.third_party)
    return raw
def tier(s):
    if s>=100:return "CRITICAL"
    if s>=40:return "HIGH"
    if s>=15:return "MODERATE"
    return "LOW"
risk_score=score(procurement)
risk_tier=tier(risk_score)
risk_score,risk_tier

## 5. Risk-based control baseline

In [ ]:
BASELINES={
 "LOW":{"agent_card","owner","basic_eval"},
 "MODERATE":{"agent_card","owner","dependency_inventory","threat_model","evaluation","monitoring"},
 "HIGH":{"agent_card","owner","dependency_inventory","threat_model","authorization_review","evaluation","red_team","monitoring","kill_switch","approval"},
 "CRITICAL":{"agent_card","owner","dependency_inventory","enhanced_threat_model","authorization_review","independent_eval","enhanced_red_team","continuous_monitoring","kill_switch","executive_approval","recertification"}
}
required=BASELINES[risk_tier]
required

## 6. Third-party and supply-chain inventory

In [ ]:
deps=pd.DataFrame([d.model_dump() for d in procurement.dependencies])
deps

In [ ]:
def supply_chain_findings(card):
    issues=[]
    for d in card.dependencies:
        if not d.version: issues.append((d.name,"UNPINNED_VERSION"))
        if d.kind in {"model","container","mcp"} and not d.digest:
            issues.append((d.name,"MISSING_INTEGRITY_EVIDENCE"))
        if d.third_party and d.trust=="LOW":
            issues.append((d.name,"LOW_TRUST_THIRD_PARTY"))
    return issues
supply_chain_findings(procurement)

## 7. Control attestations

In [ ]:
class Attestation(BaseModel):
    control_id:str
    agent_id:str
    agent_version:str
    status:Literal["PASS","FAIL","CONDITIONAL"]
    owner:str
    tester:str
    evidence_uri:str
    tested_at:datetime
    expires_at:datetime

attestations=[
 Attestation(control_id="agent_card",agent_id="AG-002",agent_version="3.2.0",status="PASS",
             owner="Procurement",tester="AI Governance",evidence_uri="evidence://card/002",
             tested_at=now(),expires_at=now()+timedelta(days=180)),
 Attestation(control_id="threat_model",agent_id="AG-002",agent_version="3.2.0",status="PASS",
             owner="Security",tester="Security Assurance",evidence_uri="evidence://tm/002",
             tested_at=now(),expires_at=now()+timedelta(days=90)),
 Attestation(control_id="evaluation",agent_id="AG-002",agent_version="3.2.0",status="PASS",
             owner="AI Engineering",tester="AI Assurance",evidence_uri="evidence://eval/002",
             tested_at=now(),expires_at=now()+timedelta(days=30))
]
pd.DataFrame([a.model_dump() for a in attestations])[["control_id","status","tester","expires_at"]]

## 8. Evidence completeness

In [ ]:
def attested_controls(agent_id,version,records):
    return {a.control_id for a in records if a.agent_id==agent_id and a.agent_version==version and a.status=="PASS" and a.expires_at>now()}
present=attested_controls(procurement.agent_id,procurement.version,attestations)
missing=required-present
present,missing

## 9. Approval routing

In [ ]:
def approval_route(tier):
    return {
      "LOW":["Product Owner"],
      "MODERATE":["Business Owner","Domain Governance"],
      "HIGH":["Business Owner","AI Governance","Risk"],
      "CRITICAL":["Business Owner","AI Governance","Risk","Executive Risk Authority"]
    }[tier]
approval_route(risk_tier)

## 10. Bounded approval

In [ ]:
class Approval(BaseModel):
    approval_id:str=Field(default_factory=lambda:"APR-"+uuid.uuid4().hex[:8])
    agent_id:str
    version:str
    risk_tier:str
    autonomy:int
    scope:str
    approver:str
    conditions:list[str]=[]
    approved_at:datetime=Field(default_factory=now)
    expires_at:datetime

approval=Approval(agent_id="AG-002",version="3.2.0",risk_tier=risk_tier,autonomy=2,
                  scope="Approved vendors; PO creation <= $15k",
                  approver="Enterprise Risk",
                  conditions=["Manager approval >= $10k"],
                  expires_at=now()+timedelta(days=90))
approval

## 11. Governance lifecycle state machine

In [ ]:
TRANSITIONS={
 "DRAFT":{"REGISTERED"},
 "REGISTERED":{"ASSESSING"},
 "ASSESSING":{"TESTING","SUSPENDED"},
 "TESTING":{"PENDING_APPROVAL","ASSESSING","SUSPENDED"},
 "PENDING_APPROVAL":{"APPROVED","TESTING","SUSPENDED"},
 "APPROVED":{"ACTIVE"},
 "ACTIVE":{"RECERTIFICATION_DUE","SUSPENDED","RETIRED","ASSESSING"},
 "RECERTIFICATION_DUE":{"ACTIVE","SUSPENDED","RETIRED"},
 "SUSPENDED":{"ASSESSING","RETIRED"},
 "RETIRED":set()
}
def transition(current,target):
    if target not in TRANSITIONS[current]: raise ValueError(f"Invalid {current} -> {target}")
    return target
transition("APPROVED","ACTIVE")

## 12. Governance-as-code

In [ ]:
def governance_checks(card,tier,records,approval=None):
    issues=[]
    issues += ownership_gate(card)
    issues += [f"SUPPLY_CHAIN:{x[0]}:{x[1]}" for x in supply_chain_findings(card)]
    current=attested_controls(card.agent_id,card.version,records)
    for c in BASELINES[tier]-current:
        issues.append(f"MISSING_CONTROL:{c}")
    if tier in {"HIGH","CRITICAL"}:
        if not approval or approval.agent_id!=card.agent_id or approval.version!=card.version or approval.expires_at<=now():
            issues.append("MISSING_OR_INVALID_APPROVAL")
    return issues
governance_checks(procurement,risk_tier,attestations,approval)[:10]

## 13. CI/CD gate

In [ ]:
def deployment_gate(card,tier,records,approval):
    findings=governance_checks(card,tier,records,approval)
    blocking=[x for x in findings if x.startswith(("MISSING_","SUPPLY_CHAIN"))]
    return {"decision":"BLOCK" if blocking else "PASS","blocking":blocking,"all_findings":findings}
gate=deployment_gate(procurement,risk_tier,attestations,approval)
gate

## 14. Simulate completed controls

In [ ]:
for c in sorted(required-present):
    attestations.append(Attestation(control_id=c,agent_id=procurement.agent_id,agent_version=procurement.version,
      status="PASS",owner="Assigned Control Owner",tester="Assurance",
      evidence_uri=f"evidence://{c}/002",tested_at=now(),expires_at=now()+timedelta(days=90)))
# add integrity evidence to the model dependency
procurement.dependencies[0].digest="sha256:model-verified"
deployment_gate(procurement,risk_tier,attestations,approval)

## 15. Material change detection

In [ ]:
MATERIALITY={
 "documentation":"NON_MATERIAL",
 "code_refactor":"NON_MATERIAL",
 "prompt":"MATERIAL",
 "model_version":"MATERIAL",
 "knowledge_source":"MATERIAL",
 "tool_version":"MATERIAL",
 "new_tool":"MAJOR",
 "new_permission":"MAJOR",
 "new_sensitive_data":"MAJOR",
 "autonomy_increase":"MAJOR",
 "irreversible_authority":"CRITICAL"
}
def route_changes(changes):
    rank={"NON_MATERIAL":0,"MATERIAL":1,"MAJOR":2,"CRITICAL":3}
    level=max((MATERIALITY.get(c,"MATERIAL") for c in changes),key=lambda x:rank[x])
    return {
      "NON_MATERIAL":"AUTOMATED_REGRESSION",
      "MATERIAL":"TARGETED_REASSESSMENT",
      "MAJOR":"RECERTIFY_AND_REAPPROVE",
      "CRITICAL":"SUSPEND_AND_FULL_REVIEW"
    }[level]
route_changes(["prompt","new_tool"])

## 16. Attestation invalidation by change

In [ ]:
CONTROL_IMPACT={
 "prompt":{"evaluation","red_team"},
 "model_version":{"evaluation","red_team"},
 "new_tool":{"threat_model","authorization_review","red_team","evaluation"},
 "new_permission":{"authorization_review","threat_model"},
 "knowledge_source":{"evaluation","red_team"},
 "autonomy_increase":{"threat_model","authorization_review","evaluation","red_team","approval"}
}
def controls_to_retest(changes):
    out=set()
    for c in changes: out |= CONTROL_IMPACT.get(c,set())
    return out
controls_to_retest(["new_tool","prompt"])

## 17. Recertification

In [ ]:
def recertify(card,approval,records,last_incident_days=None):
    findings=[]
    if not card.business_owner: findings.append("OWNER_INVALID")
    if approval.expires_at<=now(): findings.append("APPROVAL_EXPIRED")
    expired=[a.control_id for a in records if a.agent_id==card.agent_id and a.agent_version==card.version and a.expires_at<=now()]
    if expired: findings.append(f"EXPIRED_ATTESTATIONS:{expired}")
    if last_incident_days is not None and last_incident_days<30: findings.append("RECENT_INCIDENT_REVIEW_REQUIRED")
    return {"decision":"RECERTIFY" if not findings else "REVIEW","findings":findings}
recertify(procurement,approval,attestations)

## 18. Exceptions

In [ ]:
class ExceptionRecord(BaseModel):
    exception_id:str=Field(default_factory=lambda:"EX-"+uuid.uuid4().hex[:8])
    agent_id:str
    requirement:str
    reason:str
    compensating_controls:list[str]
    owner:str
    risk_acceptor:str
    expires_at:datetime

exception=ExceptionRecord(agent_id="AG-002",requirement="enhanced vendor penetration test",
                          reason="Vendor test scheduled",compensating_controls=["reduced tool scope","enhanced monitoring"],
                          owner="Procurement",risk_acceptor="Security Risk",
                          expires_at=now()+timedelta(days=21))
exception

## 19. Exception aging

In [ ]:
def exception_health(ex):
    days=(ex.expires_at-now()).days
    if days<0:return "EXPIRED"
    if days<=14:return "EXPIRING"
    return "ACTIVE"
exception_health(exception)

## 20. Incident response routing

In [ ]:
def incident_actions(kind,severity):
    base=["PRESERVE_EVIDENCE","OPEN_INCIDENT"]
    specific={
      "UNAUTHORIZED_ACTION":["REVOKE_DELEGATION","DISABLE_TOOL"],
      "DATA_LEAK":["REVOKE_CREDENTIALS","QUARANTINE_MEMORY","BLOCK_EGRESS"],
      "MCP_COMPROMISE":["DISABLE_MCP","ROTATE_CREDENTIALS","REVIEW_DEPENDENCIES"],
      "RUNAWAY_AGENT":["DISABLE_AGENT","REDUCE_AUTONOMY"]
    }.get(kind,["REDUCE_AUTONOMY"])
    if severity=="CRITICAL": specific += ["EXECUTIVE_ESCALATION","LEGAL_COMPLIANCE_REVIEW"]
    return base+specific
incident_actions("MCP_COMPROMISE","CRITICAL")

## 21. Kill-switch authority

In [ ]:
KILL_AUTHORITY={
 "DISABLE_AGENT":{"AI Platform","Security Incident Commander","Product Owner"},
 "DISABLE_TOOL":{"Tool Owner","Security Incident Commander","AI Platform"},
 "DISABLE_MCP":{"AI Platform","Security Incident Commander"},
 "REVOKE_CREDENTIALS":{"IAM","Security Incident Commander"},
 "REDUCE_AUTONOMY":{"Product Owner","AI Governance","Risk"}
}
KILL_AUTHORITY

## 22. Audit package

In [ ]:
def audit_package(card,tier,records,approval,exceptions):
    return {
      "agent_card":card.model_dump(),
      "risk":{"score":score(card),"tier":tier},
      "dependencies":[d.model_dump() for d in card.dependencies],
      "attestations":[a.model_dump() for a in records if a.agent_id==card.agent_id],
      "approval":approval.model_dump() if approval else None,
      "exceptions":[e.model_dump() for e in exceptions],
      "generated_at":now().isoformat()
    }
audit=audit_package(procurement,risk_tier,attestations,approval,[exception])
list(audit.keys())

## 23. Audit package integrity

In [ ]:
audit_digest=sha(audit)
audit_digest[:24]

## 24. Portfolio metrics

In [ ]:
portfolio=pd.DataFrame([
 {"agent":"Research Copilot","risk":"LOW","autonomy":0,"owner":True,"third_party":False,"recert_overdue":False,"open_exceptions":0},
 {"agent":"Procurement Agent","risk":"HIGH","autonomy":2,"owner":True,"third_party":True,"recert_overdue":False,"open_exceptions":1},
 {"agent":"Payment Ops Agent","risk":"CRITICAL","autonomy":3,"owner":True,"third_party":False,"recert_overdue":True,"open_exceptions":0},
 {"agent":"Vendor Support Agent","risk":"MODERATE","autonomy":1,"owner":False,"third_party":True,"recert_overdue":False,"open_exceptions":2}
])
metrics={
 "total_agents":len(portfolio),
 "high_critical":int(portfolio.risk.isin(["HIGH","CRITICAL"]).sum()),
 "third_party":int(portfolio.third_party.sum()),
 "unowned":int((~portfolio.owner).sum()),
 "recert_overdue":int(portfolio.recert_overdue.sum()),
 "open_exceptions":int(portfolio.open_exceptions.sum())
}
metrics

## 25. KRIs

In [ ]:
KRIS={
 "UNOWNED_AGENT": portfolio[~portfolio.owner].agent.tolist(),
 "OVERDUE_RECERTIFICATION": portfolio[portfolio.recert_overdue].agent.tolist(),
 "HIGH_CRITICAL_COUNT": metrics["high_critical"],
 "THIRD_PARTY_AGENT_COUNT": metrics["third_party"],
 "OPEN_EXCEPTION_COUNT": metrics["open_exceptions"]
}
KRIS

## 26. Governance drift

In [ ]:
approved_snapshot={
 "version":"3.2.0",
 "permissions":{"vendor.read","po.prepare","po.create"},
 "dependency_versions":{"foundation-model-x":"2026-07","procurement-mcp":"1.4.2"}
}
runtime_snapshot={
 "version":"3.2.0",
 "permissions":{"vendor.read","po.prepare","po.create","vendor.create"},
 "dependency_versions":{"foundation-model-x":"2026-08","procurement-mcp":"1.4.2"}
}
def drift(approved,runtime):
    findings=[]
    added=runtime["permissions"]-approved["permissions"]
    if added: findings.append(f"UNAPPROVED_PERMISSIONS:{sorted(added)}")
    if runtime["dependency_versions"]!=approved["dependency_versions"]:
        findings.append("DEPENDENCY_VERSION_DRIFT")
    return findings
drift(approved_snapshot,runtime_snapshot)

## 27. Third-party review record

In [ ]:
class ThirdPartyReview(BaseModel):
    provider:str
    service:str
    data_use_reviewed:bool
    security_reviewed:bool
    incident_terms_reviewed:bool
    change_notification:bool
    exit_plan:bool
    reviewed_at:datetime
    expires_at:datetime

vendor_review=ThirdPartyReview(provider="Vendor A",service="foundation-model-x",
 data_use_reviewed=True,security_reviewed=True,incident_terms_reviewed=True,
 change_notification=True,exit_plan=True,reviewed_at=now(),expires_at=now()+timedelta(days=180))
vendor_review

## 28. Third-party gate

In [ ]:
def third_party_gate(r):
    checks={
      "data_use":r.data_use_reviewed,
      "security":r.security_reviewed,
      "incident_terms":r.incident_terms_reviewed,
      "change_notification":r.change_notification,
      "exit_plan":r.exit_plan,
      "current":r.expires_at>now()
    }
    return {"pass":all(checks.values()),"checks":checks}
third_party_gate(vendor_review)

## 29. Example GitHub Actions governance gate

In [ ]:
github_actions = '''
name: Agent Governance Gate
on: [pull_request]

jobs:
  governance:
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v4
      - uses: actions/setup-python@v5
        with:
          python-version: "3.12"
      - run: pip install pydantic pyyaml
      - run: python governance/validate_agent_card.py agent-card.yaml
      - run: python governance/check_dependencies.py agent-card.yaml
      - run: python governance/check_attestations.py agent-card.yaml
      - run: python governance/check_approval.py agent-card.yaml
      - run: pytest tests/governance tests/security tests/evaluation
'''
print(github_actions)

## 30. Enterprise onboarding orchestrator

In [ ]:
def onboarding_summary(card,records,approval,third_party_reviews):
    s=score(card); t=tier(s)
    return {
      "agent":f"{card.agent_id}:{card.version}",
      "risk_score":s,
      "risk_tier":t,
      "ownership_findings":ownership_gate(card),
      "supply_chain_findings":supply_chain_findings(card),
      "required_controls":sorted(BASELINES[t]),
      "missing_controls":sorted(BASELINES[t]-attested_controls(card.agent_id,card.version,records)),
      "approval_route":approval_route(t),
      "third_party_reviews_pass":all(third_party_gate(r)["pass"] for r in third_party_reviews),
      "deployment_gate":deployment_gate(card,t,records,approval)["decision"]
    }
onboarding_summary(procurement,attestations,approval,[vendor_review])

## 31. Exercises

1. Add a `ThirdPartyAgent` type with provider-specific risk.
2. Add regulatory scope and geographic deployment to the agent card.
3. Add a software/model SBOM representation.
4. Verify artifact digests against an approved manifest.
5. Add Sigstore-style provenance metadata.
6. Add OpenSSF Model Signing metadata for model artifacts.
7. Build a policy that rejects unregistered MCP servers.
8. Add control attestation signatures.
9. Make attestation expiry risk-tier dependent.
10. Add event-driven recertification after incidents.
11. Add automatic recertification after permission expansion.
12. Add approval invalidation after autonomy increase.
13. Add exception approval authority by risk.
14. Detect repeated exceptions as a KRI.
15. Add a retirement workflow.
16. Generate a Markdown audit report.
17. Generate a quarterly governance portfolio report.
18. Integrate an OPA/Rego policy decision.
19. Export governance evidence using OpenTelemetry attributes.
20. Design an enterprise dashboard for the governance council.

## 32. Final design challenge

Extend the notebook into a service that accepts an `agent-card.yaml` and returns:

```json
{
  "risk_tier": "HIGH",
  "required_controls": [],
  "missing_evidence": [],
  "approval_route": [],
  "change_route": null,
  "deployment_decision": "PASS | BLOCK",
  "recertification_due": false,
  "audit_package_id": "..."
}
```

Then connect it to CI/CD.

The target state is:

> **A new or changed enterprise agent cannot reach production without a current governance record, required evidence and valid decision authority.**